In [1]:
import pandas as pd

adf = pd.read_csv('../analysis/summary-a.csv')
bdf = pd.read_csv('../analysis/summary-b.csv')
print(f"A: {len(adf)} B {len(bdf)}")

A: 2940 B 2940


In [2]:
all_attr_scores = []
all_q_values = []

def get_col_value(qnum, col, somedf):
    try:
        return somedf[somedf.question==qnum][col].iloc[0]
    except:
        return None


for i, lga in enumerate(adf.lga.unique()):
    ladf = adf[adf.lga==lga]
    lbdf = bdf[bdf.lga==lga]
    if len(ladf) != 20 or len(lbdf) != 20:
        print(f"{i+1} {lga}: {len(ladf)} {len(lbdf)}")
    for q in range(1,21):
        all_q_values.append((lga, get_col_value(q, 'state', ladf),  get_col_value(q, 'declared', ladf), q, get_col_value(q, 'positive', ladf), get_col_value(q, 'positive', lbdf), get_col_value(q, 'answer', ladf), get_col_value(q, 'answer',lbdf), get_col_value(q, 'quote', ladf), get_col_value(q, 'quote',lbdf)))

print(len(all_q_values))

2940


In [3]:
allqs = pd.DataFrame(all_q_values, columns=['lga','state','declared','question','run_a','run_b', 'answer_a','answer_b','quote_a','quote_b'])
#allqs = allqs.dropna(subset='run_b')
allqs

,lga,state,declared,question,run_a,run_b,answer_a,answer_b,quote_a,quote_b
0,Adelaide,SA,False,1,True,True,"The policy under review, the Corporate Carbon ...","The policy under review, the Corporate Carbon ...",The AHC Strategic Plan (2016) states that “we ...,The AHC Strategic Plan (2016) states that “we ...
1,Adelaide,SA,False,2,True,True,The documents do explicitly explain the need f...,The documents clearly articulate the need for ...,This plan sets out Council’s goals and targets...,This plan sets out Council’s goals and targets...
2,Adelaide,SA,False,3,False,False,The documents do not explicitly call for rapid...,The documents do not explicitly call for rapid...,NaN,NaN
3,Adelaide,SA,False,4,True,True,The documents do provide specific timeframes f...,The documents do provide specific timeframes f...,Figure 8 below shows that 100% renewable energ...,Figure 8 below shows that 100% renewable energ...
4,Adelaide,SA,False,5,False,False,The documents do not explicitly state that a c...,The documents do not explicitly state that a c...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2935,Yorke,SA,False,16,True,True,The documents indeed show an explicit intent t...,The documents indeed show an explicit intent t...,Develop state wide policies for biodiversity m...,Develop state wide policies for biodiversity m...
2936,Yorke,SA,False,17,True,True,The document does explicitly encourage buildin...,The document does explicitly encourage buildin...,Regular review and update of regional climate ...,Establish a network of regional climate change...
2937,Yorke,SA,False,18,True,True,The documents indeed refer to specific regiona...,The documents indeed refer to specific regiona...,NaN,NaN
2938,Yorke,SA,False,19,True,True,The documents do explicitly discuss the impact...,The documents do discuss the impact of climate...,NaN,NaN


In [4]:
from pallm.framework import attribute_questions, attribute_names

qtext = {}
qnum_to_attr = {}
for i, attr in enumerate(attribute_questions):
    for (qnum, qwords) in attr:
        qtext[qnum] = qwords
        qnum_to_attr[qnum] = i+1
        
qnums = sorted(qtext.keys())

def get_attr_for_question(row):
    return qnum_to_attr[row.question]

allqs['attribute'] = allqs.apply(get_attr_for_question, axis=1)
allqs

,lga,state,declared,question,run_a,run_b,answer_a,answer_b,quote_a,quote_b,attribute
0,Adelaide,SA,False,1,True,True,"The policy under review, the Corporate Carbon ...","The policy under review, the Corporate Carbon ...",The AHC Strategic Plan (2016) states that “we ...,The AHC Strategic Plan (2016) states that “we ...,1
1,Adelaide,SA,False,2,True,True,The documents do explicitly explain the need f...,The documents clearly articulate the need for ...,This plan sets out Council’s goals and targets...,This plan sets out Council’s goals and targets...,1
2,Adelaide,SA,False,3,False,False,The documents do not explicitly call for rapid...,The documents do not explicitly call for rapid...,NaN,NaN,2
3,Adelaide,SA,False,4,True,True,The documents do provide specific timeframes f...,The documents do provide specific timeframes f...,Figure 8 below shows that 100% renewable energ...,Figure 8 below shows that 100% renewable energ...,2
4,Adelaide,SA,False,5,False,False,The documents do not explicitly state that a c...,The documents do not explicitly state that a c...,NaN,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...
2935,Yorke,SA,False,16,True,True,The documents indeed show an explicit intent t...,The documents indeed show an explicit intent t...,Develop state wide policies for biodiversity m...,Develop state wide policies for biodiversity m...,9
2936,Yorke,SA,False,17,True,True,The document does explicitly encourage buildin...,The document does explicitly encourage buildin...,Regular review and update of regional climate ...,Establish a network of regional climate change...,9
2937,Yorke,SA,False,18,True,True,The documents indeed refer to specific regiona...,The documents indeed refer to specific regiona...,NaN,NaN,9
2938,Yorke,SA,False,19,True,True,The documents do explicitly discuss the impact...,The documents do discuss the impact of climate...,NaN,NaN,10


In [5]:
def get_qscores(row):
    q1 = int(row.run_a)
    if not row.run_b:
        return q1
    q2 = int(row.run_b)
    if q1 == q2:
        return q1
    else:
        return q1 # 0.5

allqs['qscore'] = allqs.apply(get_qscores, axis=1)

allqs.to_csv('../analysis/all-questions-two-runs.csv')
allqs.head()

,lga,state,declared,question,run_a,run_b,answer_a,answer_b,quote_a,quote_b,attribute,qscore
0,Adelaide,SA,False,1,True,True,"The policy under review, the Corporate Carbon ...","The policy under review, the Corporate Carbon ...",The AHC Strategic Plan (2016) states that “we ...,The AHC Strategic Plan (2016) states that “we ...,1,1
1,Adelaide,SA,False,2,True,True,The documents do explicitly explain the need f...,The documents clearly articulate the need for ...,This plan sets out Council’s goals and targets...,This plan sets out Council’s goals and targets...,1,1
2,Adelaide,SA,False,3,False,False,The documents do not explicitly call for rapid...,The documents do not explicitly call for rapid...,NaN,NaN,2,0
3,Adelaide,SA,False,4,True,True,The documents do provide specific timeframes f...,The documents do provide specific timeframes f...,Figure 8 below shows that 100% renewable energ...,Figure 8 below shows that 100% renewable energ...,2,1
4,Adelaide,SA,False,5,False,False,The documents do not explicitly state that a c...,The documents do not explicitly state that a c...,NaN,NaN,3,0


In [6]:
qdiff = allqs[(allqs.answer_b.str.len() > 0) & (allqs.run_a != allqs.run_b)]
print(f"There are {len(qdiff)} questions with difference between run A and run B, out of a total of {len(allqs)}")
print(f"-> {round(len(qdiff)/len(allqs)*100,3)}%")

There are 67 questions with difference between run A and run B, out of a total of 2940
-> 2.279%


In [7]:
qdiff.question.value_counts().sort_index()

question
1      2
2      1
3      3
5      1
6      2
7      9
8      4
9      5
10     2
12    10
13     3
14     3
15     8
16     3
17     5
18     3
19     1
20     2
Name: count, dtype: int64

In [8]:
qdiff.attribute.value_counts().sort_index()

attribute
1      3
2      3
3      3
4     13
5      5
6      2
8     24
9     11
10     3
Name: count, dtype: int64